In [10]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse
import time
import json
from datetime import datetime
import unicodedata
import re

# -----------------------
# Config
# -----------------------
START_URLS = [
    "https://www.counterpunch.org/2026/04/",
    "https://www.counterpunch.org/2026/03/",
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; CounterPunch-Scraper/1.0)"
}

ARTICLE_URL_RE = re.compile(
    r"^https://www\.counterpunch\.org/\d{4}/\d{2}/\d{2}/[^/]+/?$"
)

REQUEST_DELAY = 1.5
TARGET_ARTICLES = 200
MIN_WORD_COUNT = 150
OUTLET_NAME = "CounterPunch"

# -----------------------
# Utilities
# -----------------------
def utc_now_iso():
    return datetime.utcnow().replace(microsecond=0).isoformat() + "Z"

def compute_article_id(url):
    return urlparse(url).path.rstrip("/").split("/")[-1]

def normalize_text(text: str) -> str:
    if not text:
        return text

    text = unicodedata.normalize("NFC", text)

    replacements = {
        # quotation marks
        "“": '"', "”": '"',
        "„": '"', "‟": '"',  # additional variants
        "«": '"', "»": '"',
    
        # apostrophes
        "’": "'", "‘": "'",
        "‚": "'", "‛": "'",
        "′": "'", "‵": "'",   # prime + reversed prime
        "´": "'", "ˈ": "'",
    
        # dashes
        "–": "-", "—": "-", "―": "-",
    
        # ellipsis
        "…": "...",
    
        # spacing fixes
        "\u00a0": " ",   # non-breaking space
        "\u200b": "",    # zero-width spaces
        "\u200c": "",    # zero-width non-joiner
        "\u200d": "",
        "\ufeff": "",

        "\u00A0": " ",   # non-breaking space

        # zero-width
        "\u200B": "", 
        "\u200C": "", 
        "\u200D": "", 
        "\uFEFF": "",

        # thin-space family
        "\u2006": " ", 
        "\u2007": " ", 
        "\u2008": " ", 
        "\u2009": " ",
        "\u200A": " ", 
        "\u202F": " ",

        # word joiners
        "\u2060": "", 
        "\u2061": "", 
        "\u2062": "", 
        "\u2063": "",

        # paragraph separator → proper paragraph break
        "\u2029": "\n\n",
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    # collapse accidental multi-paragraph breaks
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text
    
def split_into_chunks(text: str):
    """
    Split text into paragraph-like chunks using double newlines
    or hard line breaks.
    """
    chunks = []

    for block in text.split("\n\n"):
        block = block.strip()
        if not block:
            continue

        # Collapse internal line breaks
        block = " ".join(line.strip() for line in block.splitlines() if line.strip())
        chunks.append(block)

    return chunks

def is_boilerplate_disclaimer(text: str) -> bool:
    text_lc = text.lower().strip()

    prefixes = (
        "this piece first appeared on",
        "this article first appeared on",
        "originally published",
        "reprinted with permission",
        "this essay originally appeared",
        "first published at",
    )

    return text_lc.startswith(prefixes)

# -----------------------
# Link extractor (UNCHANGED)
# -----------------------
def extract_article_links(html_or_url):
    # If it's HTML text, skip downloading
    if html_or_url.lstrip().startswith("<"):
        soup = BeautifulSoup(html_or_url, "html.parser")
    else:
        resp = requests.get(html_or_url, headers=HEADERS, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

    matches = []

    # This selector matches your real archive structure:
    # <div id="post-XXXXX" class="post_box ...">
    selector = "div[id^='post-'][class*='post_box'] h1.headline a"

    for post in soup.select(selector):
        href = post.get("href")

        # Keep only YYYY/MM/DD URLs
        if href and ARTICLE_URL_RE.fullmatch(href):
            matches.append(href)

    return matches

def crawl_listing_pages(start_urls):
    """
    Go through the paginated listing URLs.
    Extract ONLY YYYY/MM/DD links.
    Stop when a page has zero matching article URLs.
    """
    all_links = []

    for url in start_urls:
        articles = extract_article_links(url)

        if not articles:
            print(f"No matching links found on {url}. Stopping.")
            break

        all_links.extend(articles)

    return all_links
# -----------------------
# Pagination crawl
# -----------------------
def crawl_paginated():
    collected = []

    print("[start] Crawling paginated archive\n")

    MAX_PAGES = {
        "https://www.counterpunch.org/2026/03": 9  # normalized
    }

    for base in START_URLS:
        base_norm = base.rstrip("/")
        page = 1
        limit = MAX_PAGES.get(base_norm, None)

        while True:
            if limit is not None and page > limit:
                print(f"[pagination] Reached max page limit ({limit}) for {base_norm}")
                break

            url = base_norm if page == 1 else f"{base_norm}/page/{page}/"
            print(f"[page {page}] Fetching: {url}")

            try:
                r = requests.get(url, headers=HEADERS, timeout=20)
                r.encoding = "utf-8"
                if r.status_code != 200:
                    print(f"[page {page}] HTTP {r.status_code} — stopping")
                    break
            except Exception as e:
                print(f"[page {page}] Error: {e}")
                break

            # Pass HTML instead of URL
            links = extract_article_links(r.text)
            print(f"[page {page}] Found {len(links)} links")

            if not links:
                print("[pagination] No matching links — stopping")
                break

            for link in links:
                if link not in collected:
                    collected.append(link)

            page += 1
            time.sleep(REQUEST_DELAY)

    print(f"\n[done] Collected {len(collected)} total links\n")
    return collected

# -----------------------
# Author signature detection
# -----------------------
def is_author_signature(p, index, total_paragraphs):
    text = p.get_text(" ", strip=True)
    words = text.split()

    if index < total_paragraphs - 3:
        return False

    if len(words) > 10:
        return False

    has_br = p.find("br") is not None
    title_case_words = sum(1 for w in words if w[:1].isupper())
    has_location_comma = "," in text

    if has_br and title_case_words >= 2:
        return True

    if title_case_words >= 2 and has_location_comma:
        return True

    if 2 <= len(words) <= 4 and title_case_words == len(words):
        return True

    return False


# -----------------------
# Article extractor
# -----------------------
def extract_article(url, index, total):
    print(f"[article {index}/{total}] Fetching {url}")
    crawl_timestamp = utc_now_iso()

    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        r.encoding = "utf-8"
        if r.status_code != 200:
            return None
    except Exception:
        return None

    soup = BeautifulSoup(r.text, "html.parser")

    # -----------------------
    # Headline
    # -----------------------
    h1 = soup.find("h1", class_="headline")
    headline = normalize_text(h1.get_text(strip=True)) if h1 else None

    # -----------------------
    # Publishing date
    # -----------------------
    date_tag = soup.find("span", class_="post_date")
    publishing_date = normalize_text(date_tag.get_text(strip=True)) if date_tag else None

    # -----------------------
    # Main content
    # -----------------------
    content = soup.find("div", class_="post_content")
    if not content:
        return None

    # Remove caption/image blocks
    for caption in content.find_all("div", class_="wp-caption"):
        caption.decompose()

    p_tags = content.find_all("p")

    raw_blocks = []

    for i, p in enumerate(p_tags):
        text = p.get_text(" ", strip=True)
        text = text.replace('\\"', '"')
        if not text:
            continue

        # Normalize early
        text = normalize_text(text)

        # Skip author signature
        if is_author_signature(p, i, len(p_tags)):
            continue

        # Skip boilerplate disclaimers
        if is_boilerplate_disclaimer(text):
            continue

        raw_blocks.append(text)

    if not raw_blocks:
        return None

    # -----------------------
    # Remove wrapper paragraph duplicates
    # (if one paragraph contains the entire article)
    # -----------------------
    cleaned_blocks = []
    for i, block in enumerate(raw_blocks):
        is_wrapper = False
        for j, other in enumerate(raw_blocks):
            if i != j and other in block and len(block) > len(other):
                is_wrapper = True
                break
        if not is_wrapper:
            cleaned_blocks.append(block)

    raw_blocks = cleaned_blocks

    # Remove exact duplicates while preserving order
    seen = set()
    deduped_blocks = []
    
    for block in raw_blocks:
        normalized_block = block.strip()
        if normalized_block and normalized_block not in seen:
            deduped_blocks.append(normalized_block)
            seen.add(normalized_block)
    
    raw_blocks = deduped_blocks

    # -----------------------
    # Rebuild clean body
    # -----------------------
    body_text = "\n\n".join(raw_blocks)
    paragraphs = split_into_chunks(body_text)
    body_text = "\n\n".join(paragraphs)
    word_count = len(body_text.split())

    if word_count < MIN_WORD_COUNT:
        print(f"[article {index}] Skipped — {word_count} words")
        return None

    return {
        "article_id": compute_article_id(url),
        "url": url,
        "headline": headline,
        "publishing_date": publishing_date,
        "body": body_text,
        "word_count": word_count,
        "label": "left",
        "crawl_timestamp": crawl_timestamp,
        "outlet name": OUTLET_NAME,
    }
    
# -----------------------
# Main
# -----------------------
if __name__ == "__main__":
    print("\n=== CounterPunch Scraper Starting ===\n")

    links = crawl_paginated()
    print(f"\n[main] Collected {len(links)} article URLs\n")

    articles = []

    for i, link in enumerate(links, 1):
        if len(articles) >= TARGET_ARTICLES:
            print(f"\n[main] Reached target of {TARGET_ARTICLES} valid articles\n")
            break
    
        article = extract_article(link, i, len(links))
        if article:
            articles.append(article)
            print(f"[main] Accepted {len(articles)}/{TARGET_ARTICLES}")
    
        time.sleep(REQUEST_DELAY)

    print(f"\n[main] Extracted {len(articles)} articles")

    if articles:
        with open("counterpunch_200_articles.json", "w", encoding="utf-8") as f:
            json.dump(articles, f, indent=2, ensure_ascii=False)

        print("[main] Saved to counterpunch_200_articles.json")

    print("\n=== Done ===\n")


=== CounterPunch Scraper Starting ===

[start] Crawling paginated archive

[page 1] Fetching: https://www.counterpunch.org/2026/04
[page 1] Found 50 links
[page 2] Fetching: https://www.counterpunch.org/2026/04/page/2/
[page 2] Found 50 links
[page 3] Fetching: https://www.counterpunch.org/2026/04/page/3/
[page 3] Found 50 links
[page 4] Fetching: https://www.counterpunch.org/2026/04/page/4/
[page 4] Found 7 links
[page 5] Fetching: https://www.counterpunch.org/2026/04/page/5/
[page 5] HTTP 404 — stopping
[page 1] Fetching: https://www.counterpunch.org/2026/03
[page 1] Found 50 links
[page 2] Fetching: https://www.counterpunch.org/2026/03/page/2/
[page 2] Found 50 links
[page 3] Fetching: https://www.counterpunch.org/2026/03/page/3/
[page 3] Found 50 links
[page 4] Fetching: https://www.counterpunch.org/2026/03/page/4/
[page 4] Found 50 links
[page 5] Fetching: https://www.counterpunch.org/2026/03/page/5/
[page 5] Found 50 links
[page 6] Fetching: https://www.counterpunch.org/2026/03/